In [20]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
print(data.DESCR)

.. _california_housing_dataset:

California Housing dataset
--------------------------

**Data Set Characteristics:**

:Number of Instances: 20640

:Number of Attributes: 8 numeric, predictive attributes and the target

:Attribute Information:
    - MedInc        median income in block group
    - HouseAge      median house age in block group
    - AveRooms      average number of rooms per household
    - AveBedrms     average number of bedrooms per household
    - Population    block group population
    - AveOccup      average number of household members
    - Latitude      block group latitude
    - Longitude     block group longitude

:Missing Attribute Values: None

This dataset was obtained from the StatLib repository.
https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html

The target variable is the median house value for California districts,
expressed in hundreds of thousands of dollars ($100,000).

This dataset was derived from the 1990 U.S. census, using one row per ce

In [21]:
df = pd.DataFrame(data.data, columns=data.feature_names)

In [22]:
df['Price'] = fetch_california_housing().target

In [23]:
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [27]:
!pip install urllib3

In [31]:
from urllib.parse import urlparse

X=df.drop(columns=["Price"])
y=df["Price"]

In [37]:
def hyperparameter_tuning(X_train, y_train, model_params):
    rf = RandomForestRegressor()
    grid_search = GridSearchCV(estimator=rf, param_grid=model_params, cv=5, n_jobs=-1, verbose=2,scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    return grid_search

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [39]:
from mlflow.models.signature import infer_signature

signature = infer_signature(X_train, y_train)
print(signature)

inputs: 
  ['MedInc': double (required), 'HouseAge': double (required), 'AveRooms': double (required), 'AveBedrms': double (required), 'Population': double (required), 'AveOccup': double (required), 'Latitude': double (required), 'Longitude': double (required)]
outputs: 
  ['Price': double (required)]
params: 
  None



In [40]:
model_params = {
    'n_estimators': [100,200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

In [42]:
with mlflow.start_run():
    grid_search = hyperparameter_tuning(X_train, y_train, model_params)

    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    print(f"Best Hyperparameters: {grid_search.best_params_}")
    print(f"Mean Squared Error: {mse}")

    mlflow.log_params({
        "best_n_estimators": grid_search.best_params_['n_estimators'],
        "best_max_depth": grid_search.best_params_['max_depth'],
        "best_min_samples_split": grid_search.best_params_['min_samples_split'],
        "best_min_samples_leaf": grid_search.best_params_['min_samples_leaf']
    })
    mlflow.log_metric("mse", mse)

    mlflow.set_tracking_uri(uri="http://localhost:5000")
    traclking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    if traclking_url_type_store != "file":
        mlflow.sklearn.log_model(best_model, "model", registered_model_name="best_rf_model")
    else:
        mlflow.sklearn.log_model(best_model, "model", signature=signature)


Fitting 5 folds for each of 24 candidates, totalling 120 fits


2026/03/31 16:46:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best Hyperparameters: {'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Mean Squared Error: 0.2539369277477959


2026/03/31 16:46:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'best_rf_model'.
2026/03/31 16:47:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: best_rf_model, version 1


🏃 View run brawny-carp-291 at: http://localhost:5000/#/experiments/0/runs/dfad50500e6f40319ff55cf788c51508
🧪 View experiment at: http://localhost:5000/#/experiments/0


Created version '1' of model 'best_rf_model'.
